In [65]:
import os
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field


In [54]:
def chunk_docs(file_path, source_name):
    r"""Split a text file into sentence-based chunks and label each chunk with a source.

        Args:
            file_path: Path to the text file to read.
            source_name: Name to associate with each returned chunk, such as "Delivery Policy".

        Returns:
            A list of dictionaries where each item contains a chunk of text and its source.
            Example:
                delivery_policy = chunk_docs(r'C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_01.txt', 'Delivery Policy')
                # delivery_policy -> [{"text": "...", "source": "Delivery Policy"}, ...]
    """

    with open(file_path, 'r') as file:
        text = file.read()
    paragraph = text.strip().split(".")

    chunks = []
    for para in paragraph:
        para = para.strip()

        if len(para) < 50:
            continue
        chunks.append({"text" : para,
                    "source" : source_name})
    return chunks


In [55]:
delivery_policy = chunk_docs(r'C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_01.txt', 'Delivery Policy')
return_refund = chunk_docs(r"C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_02.txt", "Returns & Refunds")
membership_tiers = chunk_docs(r"C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_03.txt", "Membership Tiers")
order_tracking = chunk_docs(r"C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_04.txt", "Order Tracking")
order_cancellation_policy = chunk_docs(r"C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_05.txt", "Order Cancellation Policy")
damaged_missing_items = chunk_docs(r"C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_06.txt", "Damaged or Missing Items")
gift_cards = chunk_docs(r"C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_07.txt", "Gift Cards")
customer_support_hours = chunk_docs(r"C:\Users\DELL\OneDrive\Desktop\.venv\support_assistant\docs\doc_08.txt", "Customer Support Hours")

In [56]:
all_chunks = delivery_policy + return_refund + membership_tiers + order_tracking + order_cancellation_policy + damaged_missing_items + gift_cards + customer_support_hours
print(f"Number of chunks : {len(all_chunks)}")

Number of chunks : 25


In [57]:
import chromadb
chroma_client = chromadb.Client()
# Reuse the collection if it already exists; otherwise create it
collection = chroma_client.get_or_create_collection(name="Zepto_Document_corpus")



In [58]:
documents, ids, metadatas = [],[],[]

for i, chunk in enumerate(all_chunks):
  documents.append(chunk["text"])
  ids.append(f"chunk_{i}")
  metadatas.append({"source" : chunk['source']})
collection.add(documents= documents, ids = ids, metadatas= metadatas)

In [62]:
prompt_template = """You are a helpful customer support assistant for Zepto.\n\nUse the following provided context about Zepto's policies to answer the user's question.\n\nAnswer the user's question accurately and concisely based *only* on the provided context.\n\nProvide a direct answer. If the answer requires multiple points, use a bulleted list.\n\nKeep the answer to a maximum of 3 sentences.\n\nDo not use any information outside of the provided context. If the answer is not present in the context, state that you don't have enough information.\n\nFew-shot Example:\nUser: How long are gift cards valid?\nContext: "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, a...by email or SMS within minutes of purchase", "Gift cards are valid for 1 year from the date of issue and carry no maintenance fees", "Gift card balance can be combined with one other payment method at checkout but canno... another gift card in the same transaction"\nAssistant: Zepto gift cards are valid for 1 year from the date of issue and carry no maintenance fees.\n\nUser: {question}\nContext: {context}\nAssistant:"""

In [64]:
# Set MOCK_LLM. For grading, this should be 1 or unset.
# For optional real LLM extension, set to 0.
MOCK_LLM = os.environ.get("MOCK_LLM", "1") == "1"

# Define the state
class AgentState(TypedDict):
    question: str
    context: List[str]
    answer: str
    intent: str # 'policy_question' or 'general_question'

# --- Nodes Implementation ---

def classify_intent(state: AgentState) -> AgentState:
    """Classifies the incoming query as 'policy_question' or 'general_question'."""
    question = state["question"].lower()
    intent = "general_question" # Default

    if MOCK_LLM:
        # Mock mode: keyword heuristic
        policy_keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
        if any(keyword in question for keyword in policy_keywords):
            intent = "policy_question"
    else:
        # Optional MOCK_LLM=0 extension: call the LLM to classify instead.
        # Placeholder for real LLM classification logic.
        print("MOCK_LLM is 0: Calling LLM for intent classification (placeholder)")
        # In a real scenario, integrate an LLM call here, e.g.:
        # llm_classifier_chain.invoke(question)
        # For the graded baseline, we will simulate the mock behavior here as well if LLM is not integrated.
        policy_keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
        if any(keyword in question for keyword in policy_keywords):
            intent = "policy_question"

    print(f"Classified intent: {intent}")
    return {**state, "intent": intent}

def retrieve_and_answer(state: AgentState) -> AgentState:
    """Retrieves relevant documents and generates an answer for policy questions."""
    question = state["question"]
    print(f"Retrieving for question: {question}")

    # Retrieval step (always real, as embedding and ChromaDB need no API key)
    # Access the global chromadb collection
    global collection
    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    retrieved_docs = [doc for doc in results['documents'][0]]
    retrieved_metadatas = results['metadatas'][0]

    # Combine document and source for context
    context_list = []
    for doc, meta in zip(retrieved_docs, retrieved_metadatas):
        context_list.append(f"Source: {meta.get('source', 'Unknown')}\nText: {doc}")

    answer = ""
    if MOCK_LLM:
        # Mock mode: canned templated answer
        if retrieved_docs:
            top_chunk_snippet = retrieved_docs[0][:200] # First ~200 characters
            answer = f"Based on the retrieved context: {top_chunk_snippet}..."
        else:
            answer = "I could not find relevant information in the knowledge base."
    else:
        # Optional MOCK_LLM=0 extension: prompt the real LLM.
        # Placeholder for real LLM answer generation logic using the structured template.
        print("MOCK_LLM is 0: Calling LLM for answer generation (placeholder)")
        global prompt_template
        if retrieved_docs:
            formatted_context = "\n\n".join(context_list)
            # In a real scenario, integrate an LLM call here, e.g.:
            # from langchain_openai import ChatOpenAI
            # from langchain_core.prompts import ChatPromptTemplate
            # from langchain_core.output_parsers import StrOutputParser
            # llm = ChatOpenAI(model="gpt-4", temperature=0)
            # prompt = ChatPromptTemplate.from_template(prompt_template)
            # chain = prompt | llm | StrOutputParser()
            # answer = chain.invoke({"question": question, "context": formatted_context})
            answer = f"Real LLM would answer here based on context: {formatted_context[:100]}..."
        else:
            answer = "Real LLM could not find information based on retrieval."

    print(f"Generated answer: {answer}")
    return {**state, "context": context_list, "answer": answer}

def direct_answer(state: AgentState) -> AgentState:
    """Generates a direct answer for general questions."""
    question = state["question"]
    answer = ""

    if MOCK_LLM:
        # Mock mode: fixed canned string
        answer = "I can only answer questions about Zepto policies right now."
    else:
        # Optional MOCK_LLM=0 extension: prompt the LLM directly.
        # Placeholder for real LLM direct answer logic.
        print("MOCK_LLM is 0: Calling LLM for direct answer (placeholder)")
        # In a real scenario, integrate an LLM call here, e.g.:
        # from langchain_openai import ChatOpenAI
        # from langchain_core.prompts import ChatPromptTemplate
        # from langchain_core.output_parsers import StrOutputParser
        # llm = ChatOpenAI(model="gpt-4", temperature=0)
        # prompt = ChatPromptTemplate.from_template("Answer the following question: {question}")
        # chain = prompt | llm | StrOutputParser()
        # answer = chain.invoke({"question": question})
        answer = f"Real LLM would answer the general question: {question}"

    print(f"Generated direct answer: {answer}")
    return {**state, "answer": answer}

# --- Conditional Edge Logic ---

def route_intent(state: AgentState) -> str:
    """Routes based on the classified intent."""
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    elif state["intent"] == "general_question":
        return "direct_answer"
    else:
        raise ValueError(f"Unknown intent: {state['intent']}")

# --- Build the Graph ---

workflow = StateGraph(AgentState)

workflow.add_node("classify_intent", classify_intent)
workflow.add_node("retrieve_and_answer", retrieve_and_answer)
workflow.add_node("direct_answer", direct_answer)

workflow.set_entry_point("classify_intent")

workflow.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

workflow.add_edge("retrieve_and_answer", END)
workflow.add_edge("direct_answer", END)

# Compile the graph
app = workflow.compile()

print("LangGraph StateGraph 'app' has been built and compiled.")

LangGraph StateGraph 'app' has been built and compiled.


In [66]:
# Set MOCK_LLM. For grading, this should be 1 or unset.
# For optional real LLM extension, set to 0.
MOCK_LLM = os.environ.get("MOCK_LLM", "1") == "1"

# --- Pydantic Output Model ---
class Response(BaseModel):
    answer: str = Field(description="The generated answer to the user's question.")
    sources: List[str] = Field(description="List of document IDs (chunk_IDs) used to generate the answer. Empty for general questions.")
    confidence: float = Field(description="Confidence score of the answer, ranging from 0.0 to 1.0.")

# Define the state
class AgentState(TypedDict):
    question: str
    context: List[str] # This holds the text of the retrieved chunks
    answer: str # This holds the raw string answer from the LLM or mock
    intent: str # 'policy_question' or 'general_question'
    retrieved_ids: List[str] # To store IDs of retrieved chunks
    final_confidence: float # To store the confidence score


# --- Nodes Implementation ---

def classify_intent(state: AgentState) -> AgentState:
    """Classifies the incoming query as 'policy_question' or 'general_question'."""
    question = state["question"].lower()
    intent = "general_question" # Default

    if MOCK_LLM:
        # Mock mode: keyword heuristic
        policy_keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
        if any(keyword in question for keyword in policy_keywords):
            intent = "policy_question"
    else:
        # Optional MOCK_LLM=0 extension: call the LLM to classify instead.
        # Placeholder for real LLM classification logic.
        print("MOCK_LLM is 0: Calling LLM for intent classification (placeholder)")
        # For the graded baseline, we will simulate the mock behavior here as well if LLM is not integrated.
        policy_keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
        if any(keyword in question for keyword in policy_keywords):
            intent = "policy_question"

    print(f"Classified intent: {intent}")
    return {**state, "intent": intent}

def retrieve_and_answer(state: AgentState) -> AgentState:
    """Retrieves relevant documents and generates an answer for policy questions."""
    question = state["question"]
    print(f"Retrieving for question: {question}")

    # Retrieval step (always real, as embedding and ChromaDB need no API key)
    global collection
    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    retrieved_docs = [doc for doc in results['documents'][0]]
    retrieved_chunk_ids = [doc_id for doc_id in results['ids'][0]] # Get the actual chunk IDs
    retrieved_metadatas = results['metadatas'][0]

    # Combine document and source for context
    context_list = []
    for doc, meta in zip(retrieved_docs, retrieved_metadatas):
        context_list.append(f"Source: {meta.get('source', 'Unknown')}\nText: {doc}")

    answer = ""
    confidence = 1.0 # Default for mock mode

    if MOCK_LLM:
        # Mock mode: canned templated answer
        if retrieved_docs:
            top_chunk_snippet = retrieved_docs[0][:200] # First ~200 characters
            answer = f"Based on the retrieved context: {top_chunk_snippet}..."
        else:
            answer = "I could not find relevant information in the knowledge base."
            retrieved_chunk_ids = [] # No sources if no docs found
    else:
        # Optional MOCK_LLM=0 extension: prompt the real LLM.
        # Placeholder for real LLM answer generation logic using the structured template.
        print("MOCK_LLM is 0: Calling LLM for answer generation (placeholder)")
        global prompt_template
        if retrieved_docs:
            formatted_context = "\n\n".join(context_list)
            # LLM call, parsing, and retry logic would go here.
            # For demonstration, we'll mimic the mock output.
            answer = f"Real LLM would answer here based on context: {formatted_context[:100]}..."
        else:
            answer = "Real LLM could not find information based on retrieval."
            retrieved_chunk_ids = []

    print(f"Generated answer: {answer}")
    return {**state, "context": context_list, "answer": answer, "retrieved_ids": retrieved_chunk_ids, "final_confidence": confidence}


def direct_answer(state: AgentState) -> AgentState:
    """Generates a direct answer for general questions."""
    question = state["question"]
    answer = ""
    retrieved_ids = [] # No sources for general questions
    confidence = 1.0 # Default for mock mode

    if MOCK_LLM:
        # Mock mode: fixed canned string
        answer = "I can only answer questions about Zepto policies right now."
    else:
        # Optional MOCK_LLM=0 extension: prompt the LLM directly.
        # Placeholder for real LLM direct answer logic.
        print("MOCK_LLM is 0: Calling LLM for direct answer (placeholder)")
        # LLM call, parsing, and retry logic would go here.
        # For demonstration, we'll mimic the mock output.
        answer = f"Real LLM would answer the general question: {question}"

    print(f"Generated direct answer: {answer}")
    return {**state, "answer": answer, "retrieved_ids": retrieved_ids, "final_confidence": confidence}

def format_final_response(state: AgentState) -> Response:
    """Formats the final answer into the Pydantic Response model."""
    final_answer = state.get("answer", "No answer generated.")
    final_sources = state.get("retrieved_ids", [])
    final_confidence = state.get("final_confidence", 0.0)

    # In a real MOCK_LLM=0 scenario, this is where LLM output parsing and validation
    # would happen with retries. For mock, we simply construct it from state.
    # The retry logic would wrap the LLM call *within* retrieve_and_answer or direct_answer.
    # For the graded baseline, we just assume the values are set correctly by the mock nodes.
    return Response(
        answer=final_answer,
        sources=final_sources,
        confidence=final_confidence
    )

# --- Conditional Edge Logic ---

def route_intent(state: AgentState) -> str:
    """Routes based on the classified intent."""
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    elif state["intent"] == "general_question":
        return "direct_answer"
    else:
        raise ValueError(f"Unknown intent: {state['intent']}")

# --- Build the Graph ---

workflow = StateGraph(AgentState)

workflow.add_node("classify_intent", classify_intent)
workflow.add_node("retrieve_and_answer", retrieve_and_answer)
workflow.add_node("direct_answer", direct_answer)
workflow.add_node("format_final_response", format_final_response) # Add new node

workflow.set_entry_point("classify_intent")

workflow.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

# Wire the answer nodes to the new format_final_response node, which then leads to END
workflow.add_edge("retrieve_and_answer", "format_final_response")
workflow.add_edge("direct_answer", "format_final_response")
workflow.add_edge("format_final_response", END)

# Compile the graph
app = workflow.compile()

print("LangGraph StateGraph 'app' has been built and compiled with Pydantic output schema.")

LangGraph StateGraph 'app' has been built and compiled with Pydantic output schema.


In [67]:
# Example 1: Policy Question
policy_question = "What are the delivery fees?"
initial_state_policy = AgentState(question=policy_question, context=[], answer="", intent="", retrieved_ids=[], final_confidence=0.0)
final_state_policy = app.invoke(initial_state_policy)

# Construct the Response object from the final_state_policy dictionary for proper printing
result_policy_pydantic = Response(
    answer=final_state_policy.get("answer", "No answer generated."),
    sources=final_state_policy.get("retrieved_ids", []),
    confidence=final_state_policy.get("final_confidence", 0.0)
)
print(f"\n--- Policy Question Result ---\nQuestion: {policy_question}\nResult: {result_policy_pydantic.model_dump_json(indent=2)}")

# Example 2: General Question
general_question = "Hello, how are you?"
initial_state_general = AgentState(question=general_question, context=[], answer="", intent="", retrieved_ids=[], final_confidence=0.0)
final_state_general = app.invoke(initial_state_general)

# Construct the Response object from the final_state_general dictionary for proper printing
result_general_pydantic = Response(
    answer=final_state_general.get("answer", "No answer generated."),
    sources=final_state_general.get("retrieved_ids", []),
    confidence=final_state_general.get("final_confidence", 0.0)
)
print(f"\n--- General Question Result ---\nQuestion: {general_question}\nResult: {result_general_pydantic.model_dump_json(indent=2)}")

Classified intent: policy_question
Retrieving for question: What are the delivery fees?
Generated answer: Based on the retrieved context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee...

--- Policy Question Result ---
Question: What are the delivery fees?
Result: {
  "answer": "Based on the retrieved context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee...",
  "sources": [
    "chunk_1",
    "chunk_7",
    "chunk_8"
  ],
  "confidence": 1.0
}
Classified intent: general_question
Generated direct answer: I can only answer questions about Zepto policies right now.

--- General Question Result ---
Question: Hello, how are you?
Result: {
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}


In [ ]:
# Install FastAPI and Uvicorn
!pip install fastapi uvicorn requests

In [69]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio

# Pydantic request model for the API endpoint
class AskRequest(BaseModel):
    query: str

app_fastapi = FastAPI()

@app_fastapi.post("/ask", response_model=Response)
async def ask_agent(request_data: AskRequest):
    initial_state = AgentState(
        question=request_data.query,
        context=[],
        answer="",
        intent="",
        retrieved_ids=[],
        final_confidence=0.0
    )
    
    # Invoke the LangGraph agent
    final_state = app.invoke(initial_state)
    
    # Manually create the Pydantic Response object from the final_state dictionary
    # This is necessary because app.invoke() returns the state dictionary directly, 
    # not the Pydantic model that `format_final_response` creates within the graph.
    final_response_pydantic = Response(
        answer=final_state.get("answer", "No answer generated."),
        sources=final_state.get("retrieved_ids", []),
        confidence=final_state.get("final_confidence", 0.0)
    )
    
    return final_response_pydantic

print("FastAPI app 'app_fastapi' created with /ask endpoint.")

FastAPI app 'app_fastapi' created with /ask endpoint.


In [70]:
# Run the FastAPI app with uvicorn
# This will run in the background in Colab thanks to nest_asyncio

nest_asyncio.apply()

# Use a different port if 8000 is already in use
# Note: This will block the cell execution until interrupted.
# For demonstration, we'll keep it running. In a real scenario, you'd deploy this.

# To run in Colab, you might need to use a tool like ngrok for public access
# For local testing within Colab, direct localhost calls might work.
# uvicorn.run(app_fastapi, host="0.0.0.0", port=8000)

# Running uvicorn in a separate thread for non-blocking execution in Colab
import threading
import time

def run_uvicorn():
    uvicorn.run(app_fastapi, host="0.0.0.0", port=8000)

uvicorn_thread = threading.Thread(target=run_uvicorn)
uvicorn_thread.daemon = True # Allow the program to exit even if the thread is still running
uvicorn_thread.start()

print("FastAPI application started on http://0.0.0.0:8000. Waiting for it to spin up...")
time.sleep(5) # Give uvicorn a moment to start
print("Uvicorn is likely running.")

FastAPI application started on http://0.0.0.0:8000. Waiting for it to spin up...


INFO:     Started server process [2544]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Uvicorn is likely running.


In [71]:
import requests
import json

base_url = "http://127.0.0.1:8000"

# Example 1: Policy Question (should trigger retrieval)
policy_query = "How much is delivery?"
response_policy = requests.post(f"{base_url}/ask", json={"query": policy_query})

print(f"\n--- API Call: Policy Question ---")
print(f"Request Query: {policy_query}")
print(f"Status Code: {response_policy.status_code}")
if response_policy.status_code == 200:
    print("Response Body (JSON):")
    print(json.dumps(response_policy.json(), indent=2))
else:
    print(f"Error: {response_policy.text}")


# Example 2: General Question (should not trigger retrieval)
general_query = "What is the weather today?"
response_general = requests.post(f"{base_url}/ask", json={"query": general_query})

print(f"\n--- API Call: General Question ---")
print(f"Request Query: {general_query}")
print(f"Status Code: {response_general.status_code}")
if response_general.status_code == 200:
    print("Response Body (JSON):")
    print(json.dumps(response_general.json(), indent=2))
else:
    print(f"Error: {response_general.text}")

Classified intent: policy_question
Retrieving for question: How much is delivery?
Generated answer: Based on the retrieved context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee...
INFO:     127.0.0.1:49464 - "POST /ask HTTP/1.1" 200 OK

--- API Call: Policy Question ---
Request Query: How much is delivery?
Status Code: 200
Response Body (JSON):
{
  "answer": "Based on the retrieved context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee...",
  "sources": [
    "chunk_1",
    "chunk_11",
    "chunk_7"
  ],
  "confidence": 1.0
}
Classified intent: general_question
Generated direct answer: I can only answer questions about Zepto policies right now.
INFO:     127.0.0.1:49465 - "POST /ask HTTP/1.1" 200 OK

--- API Call: General Question ---
Request Query: What is the weather today?
Status Code: 200
Response Body (JSON):
{
  "answer": "I can only answer questions 